In [1]:
# 1. 외부 모듈 자동 새로고침 설정 (loader.py 수정 시 즉각 반영)
%load_ext autoreload
%autoreload 2

# 2. 필수 라이브러리 임포트
import os
import json
import pandas as pd
import FinanceDataReader as fdr
import pykrx
import OpenDartReader
import matplotlib
import seaborn
import scipy
from datetime import date

# 3. 직접 만든 로컬 모듈 임포트
from data.loader import QuantDataLoader

print("✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!")

KRX 로그인 시도...
  로그인 ID: forscom
KRX 로그인 완료.
  로그인 시간: 2026-07-28 17:15:56
  만료 시간: 2026-07-28 18:15:56
✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!


In [3]:
def test_data_loader():
    print("==================================================")
    print("🚀 QuantDataLoader 테스트를 시작합니다...")
    print("==================================================\n")
    
    # 1. 로더 인스턴스 생성
    try:
        print("[테스트 1] 로더 인스턴스화 및 환경변수 확인")
        loader = QuantDataLoader(use_cache=True)
        print("✅ 성공: DART API 키 및 로더 초기화 완료!\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")
        return

    # 2. 유니버스 로드 테스트 (Point-in-Time)
    test_date = date(2023, 7, 24)
    print(f"[테스트 2] KOSPI 유니버스 데이터 로드 ({test_date})")
    try:
        universe_df = loader.get_kospi_universe(test_date)
        print(f"✅ 성공: 총 {len(universe_df)}개 종목 로드 완료!")
        print("-" * 50)
        display(universe_df.head())
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    # 3. DART 재무제표 파싱 테스트
    ticker_to_test = '005930'
    target_year = 2023
    print(f"[테스트 3] {ticker_to_test} {target_year}년 사업보고서(11011) 파싱")
    try:
        financials = loader.parse_standardized_financials(ticker_to_test, target_year, '11011')
        print(f"✅ 성공: 재무 데이터 표준화 완료!")
        print("-" * 50)
        for key, value in financials.items():
            if pd.isna(value):
                print(f"{key:>20} : NaN")
            else:
                print(f"{key:>20} : {value:,.0f}")
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    print("==================================================")
    print("🎯 모든 테스트가 종료되었습니다.")
    print("==================================================")

# 테스트 실행
test_data_loader()

🚀 QuantDataLoader 테스트를 시작합니다...

[테스트 1] 로더 인스턴스화 및 환경변수 확인
✅ 성공: DART API 키 및 로더 초기화 완료!

[테스트 2] KOSPI 유니버스 데이터 로드 (2023-07-24)
✅ 성공: 총 834개 종목 로드 완료!
--------------------------------------------------


,ticker,name,sector,close_price,market_cap
0,005930,삼성전자,통신 및 방송 장비 제조업,70400,420272691520000
1,373220,LG에너지솔루션,일차전지 및 이차전지 제조업,597000,139698000000000
2,000660,SK하이닉스,반도체 제조업,114000,82992269610000
3,005490,POSCO홀딩스,1차 철강 제조업,642000,54294729660000
4,207940,삼성바이오로직스,기초 의약물질 제조업,742000,52811108000000


--------------------------------------------------

[테스트 3] 005930 2023년 사업보고서(11011) 파싱
✅ 성공: 재무 데이터 표준화 완료!
--------------------------------------------------
             revenue : 258,935,494,000,000
                cogs : 180,388,580,000,000
        gross_profit : 78,546,914,000,000
                 sga : 71,979,938,000,000
           inventory : 51,625,874,000,000
    operating_income : 6,566,976,000,000
          net_income : 15,487,100,000,000
 operating_cash_flow : 44,137,427,000,000
--------------------------------------------------

🎯 모든 테스트가 종료되었습니다.


In [4]:
# config.json 파라미터 불러오기
try:
    with open('config.json', 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    target_market = config['strategy_params']['market'] # 예: 'KOSPI'
    top_n = config['strategy_params']['top_n_mcap']
except FileNotFoundError:
    print("⚠️ config.json 파일이 없습니다. 기본값으로 진행합니다.")
    target_market = 'KOSPI'
    top_n = 10

print(f"🔍 {target_market} 시장 데이터를 불러오는 중...\n")

# FinanceDataReader를 통한 KRX 전종목 리스팅 조회
df_krx = fdr.StockListing('KRX')

# 지정한 시장 필터링 및 시가총액(MarCap) 기준 정렬
top_mcap_df = df_krx[df_krx['Market'] == target_market].sort_values(by='Marcap', ascending=False).head(top_n)

# 보기 좋게 컬럼명 정리
top_mcap_df = top_mcap_df[['Code', 'Name', 'Close', 'Marcap', 'Stocks']].rename(
    columns={
        'Code': '종목코드',
        'Name': '종목명',
        'Close': '종가',
        'Marcap': '시가총액',
        'Stocks': '상장주식수'
    }
)

print("✅ 정상적으로 데이터를 불러왔습니다!")
display(top_mcap_df)

🔍 KOSPI 시장 데이터를 불러오는 중...

✅ 정상적으로 데이터를 불러왔습니다!


,종목코드,종목명,종가,시가총액,상장주식수
0,005930,삼성전자,230000,1344644079840000,5846278608
1,000660,SK하이닉스,1616000,1151727021840000,712702365
2,005935,삼성전자우,164200,131749351532600,802371203
3,402340,SK스퀘어,964000,127207884104000,131958386
4,009150,삼성전기,1141000,85225507136000,74693696
5,005380,현대차,374000,76579404484000,204757766
6,373220,LG에너지솔루션,317500,74295000000000,234000000
7,207940,삼성바이오로직스,1559000,72167592609000,46290951
8,105560,KB금융,170600,60509727420400,354687734
9,032830,삼성생명,286000,57200000000000,200000000


In [1]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader

def verify_new_features():
    print("==================================================")
    print("🚀 QuantDataLoader 신규 기능 검증을 시작합니다...")
    print("==================================================\n")
    
    try:
        loader = QuantDataLoader(use_cache=True)
    except Exception as e:
        print(f"❌ 초기화 실패: {e}")
        return

    # ---------------------------------------------------------
    # 검증 1: 시계열 주가/거래량 데이터 (OHLCV) 및 캐싱
    # ---------------------------------------------------------
    print("[검증 1] get_historical_ohlcv 작동 확인")
    ticker = '005930'
    start = date(2025, 1, 1)
    end = date(2025, 6, 30)
    
    try:
        ohlcv_df = loader.get_historical_ohlcv(ticker, start, end)
        if ohlcv_df is not None and not ohlcv_df.empty:
            print(f"✅ 성공: {start} ~ {end} 시계열 데이터 {len(ohlcv_df)}일치 로드 완료")
            print(ohlcv_df[['Close', 'Volume']].head(3).to_string())
        else:
            print("❌ 실패: 데이터가 비어 있습니다.")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 2: 확장된 계정과목 매핑 확인 (자산, 부채, 자본 등)
    # ---------------------------------------------------------
    print("[검증 2] 확장된 재무제표 계정 파싱 확인 (2025년 사업보고서)")
    try:
        fin_annual = loader.parse_standardized_financials(ticker, 2025, '11011')
        keys_to_check = ['total_assets', 'total_liabilities', 'total_equity', 'interest_expense']
        
        missing = [k for k in keys_to_check if pd.isna(fin_annual.get(k, float('nan')))]
        if not missing:
            print("✅ 성공: 자산/부채/자본/이자비용 모두 정상 파싱 완료")
            for k in keys_to_check:
                print(f"   - {k}: {fin_annual[k]:,.0f}")
        else:
            print(f"⚠️ 주의: 다음 계정 누락 (정상일 수도 있음) -> {missing}")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 3: 분기 단독값 차분(Isolation) 로직 확인
    # ---------------------------------------------------------
    print("[검증 3] get_isolated_quarterly_financials 차분 로직 확인")
    try:
        # 1분기(누적)와 2분기(차분)의 매출액 비교
        q1_data = loader.get_isolated_quarterly_financials(ticker, 2025, 1)
        q2_isolated = loader.get_isolated_quarterly_financials(ticker, 2025, 2)
        
        print("✅ 성공: 1분기 및 2분기(단독) 데이터 추출 완료")
        print(f"   - 1Q 매출액 (누적=단독) : {q1_data.get('revenue', 0):,.0f}")
        print(f"   - 2Q 매출액 (차분 적용) : {q2_isolated.get('revenue', 0):,.0f}")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 4: DART API Rate Limit 카운터 작동 확인
    # ---------------------------------------------------------
    print("[검증 4] DART API 카운터 및 Rate Limit 방어벽 확인")
    try:
        current_calls = loader.dart_call_count
        print(f"✅ 성공: 현재 세션 API 호출 횟수 정상 트래킹 중 -> {current_calls}회")
        
        # 임의로 한도를 초과시켜 방어벽 테스트
        loader.dart_daily_limit = current_calls  
        
        # 💡 API를 강제로 호출하도록 일시적으로 캐시 기능 비활성화
        loader.use_cache = False 
        
        try:
            loader.get_financial_statements(ticker, 2024, '11011')
            print("❌ 실패: 한도 초과 상황에서 Exception이 발생하지 않고 통과됨!")
        except Exception as limit_err:
            print(f"✅ 성공: 방어벽 정상 작동 확인 -> {limit_err}")
            
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("==================================================\n")


if __name__ == "__main__":
    verify_new_features()

KRX 로그인 시도...
  로그인 ID: forscom
KRX 로그인 완료.
  로그인 시간: 2026-07-28 13:42:19
  만료 시간: 2026-07-28 14:42:19
🚀 QuantDataLoader 신규 기능 검증을 시작합니다...

[검증 1] get_historical_ohlcv 작동 확인
✅ 성공: 2025-01-01 ~ 2025-06-30 시계열 데이터 118일치 로드 완료
            Close    Volume
Date                       
2025-01-02  53400  16630538
2025-01-03  54400  19318046
2025-01-06  55900  19034284
--------------------------------------------------

[검증 2] 확장된 재무제표 계정 파싱 확인 (2025년 사업보고서)
✅ 성공: 자산/부채/자본/이자비용 모두 정상 파싱 완료
   - total_assets: 566,942,110,000,000
   - total_liabilities: 130,621,773,000,000
   - total_equity: 436,320,337,000,000
   - interest_expense: 11,733,764,000,000
--------------------------------------------------

[검증 3] get_isolated_quarterly_financials 차분 로직 확인
✅ 성공: 1분기 및 2분기(단독) 데이터 추출 완료
   - 1Q 매출액 (누적=단독) : 79,140,503,000,000
   - 2Q 매출액 (차분 적용) : 74,566,317,000,000
--------------------------------------------------

[검증 4] DART API 카운터 및 Rate Limit 방어벽 확인
✅ 성공: 현재 세션 API 호출 횟수 정상 트래킹 중 -> 0회
✅ 성공:

In [2]:
import pandas as pd
import numpy as np

# 프로젝트 루트 경로에서 모듈 임포트
from stages.stage1_neglected_sector import NeglectedSectorScreener

# ---------------------------------------------------------
# 1. 가상의 params.yaml 파라미터 셋업
# ---------------------------------------------------------
mock_params = {
    'stage1_return_weight': 0.6,  # 수익률 소외도에 약간 더 높은 가중치
    'stage1_volume_weight': 0.4,  # 거래대금 소외도 가중치
    'stage1_pass_ratio': 0.5      # 테스트를 위해 상위 50% 통과로 설정
}

# ---------------------------------------------------------
# 2. 가상의 섹터 데이터 생성 
# 포트폴리오 비중을 고려하여 성장주 섹터와 배당주 섹터 혼합 구성
# ---------------------------------------------------------
mock_data = {
    'sector': [
        'IT/소프트웨어',    # 지속적인 성장이 기대되나 현재 일시적 소외 (통과 예상)
        '바이오/헬스케어',  # 최근 낙폭이 급격히 가팔라진 섹터 (밸류트랩 예상)
        '고배당/금융',      # 최근 수익률과 거래대금이 양호한 섹터 (탈락 예상)
        '2차전지',          # 장기 하락 중이나 하락세가 진정된 섹터
        '건설/기계',        # 거래대금이 급감한 섹터
        '필수소비재'        # 방어주 성격의 섹터
    ],
    # return_1m: 최근 1개월 수익률 / return_6m: 최근 6개월 수익률
    'return_1m': [-0.02, -0.15, 0.03, -0.01, -0.05, 0.01], 
    'return_6m': [-0.24, -0.06, 0.08, -0.30, -0.12, 0.02], 
    # vol_prop_1m: 최근 1개월 거래대금 비중 / vol_prop_1y: 1년 평균 거래대금 비중
    'vol_prop_1m': [0.05, 0.08, 0.15, 0.06, 0.02, 0.10],   
    'vol_prop_1y': [0.15, 0.10, 0.12, 0.20, 0.06, 0.09]    
}
sector_df = pd.DataFrame(mock_data)

# ---------------------------------------------------------
# 3. 스크리너 초기화 및 실행
# ---------------------------------------------------------
screener = NeglectedSectorScreener(params=mock_params)
passed_sectors = screener.run(sector_df)

# ---------------------------------------------------------
# 4. 검증 결과 출력
# ---------------------------------------------------------
print("📊 [원본 섹터 데이터]")
display(sector_df)

print("\n🚀 [Stage 1 필터링 통과 결과 (상위 50%)]")
display(passed_sectors)

# 밸류트랩 경고가 제대로 작동했는지 검증
print("\n🔍 [밸류트랩 방어선 작동 확인]")
value_traps = passed_sectors[passed_sectors['is_value_trap_warning'] == True]
if not value_traps.empty:
    print(f"⚠️ 경고: 다음 섹터는 낙폭이 가팔라지고 있습니다 (Value Trap 위험)\n -> {value_traps['sector'].tolist()}")
else:
    print("✅ 통과된 섹터 중 급격한 낙폭 가속(Value Trap) 위험이 감지된 섹터는 없습니다.")

📊 [원본 섹터 데이터]


,sector,return_1m,return_6m,vol_prop_1m,vol_prop_1y
0,IT/소프트웨어,-0.02,-0.24,0.05,0.15
1,바이오/헬스케어,-0.15,-0.06,0.08,0.10
2,고배당/금융,0.03,0.08,0.15,0.12
3,2차전지,-0.01,-0.30,0.06,0.20
4,건설/기계,-0.05,-0.12,0.02,0.06
5,필수소비재,0.01,0.02,0.10,0.09



🚀 [Stage 1 필터링 통과 결과 (상위 50%)]


,sector,return_z_score,volume_z_score,composite_score,is_value_trap_warning
3,2차전지,1.336087,0.909633,1.165505,False
0,IT/소프트웨어,0.928467,0.831479,0.889672,False
4,건설/기계,0.113228,0.831479,0.400528,True



🔍 [밸류트랩 방어선 작동 확인]
⚠️ 경고: 다음 섹터는 낙폭이 가팔라지고 있습니다 (Value Trap 위험)
 -> ['건설/기계']


In [3]:
import pandas as pd
import numpy as np
import yaml
from datetime import date
from dateutil.relativedelta import relativedelta

# 프로젝트 모듈 임포트
from data.loader import QuantDataLoader
from stages.stage1_neglected_sector import NeglectedSectorScreener

def run_real_data_test():
    print("==================================================")
    print("🚀 실데이터 기반 Stage 1: 소외 섹터 발굴 테스트")
    print("==================================================\n")

    # 1. 로더 및 스크리너 초기화
    loader = QuantDataLoader(use_cache=True)

    # config/params.yaml에서 파라미터 로드
    try:
        with open("config/params.yaml", "r", encoding="utf-8") as f:
            yaml_config = yaml.safe_load(f)
            params = yaml_config.get('stage1_neglected_sector', {})
            print("✅ params.yaml 로드 완료")
    except Exception as e:
        print("⚠️ params.yaml 로드 실패. 기본 파라미터로 진행합니다.")
        params = {
            'stage1_return_weight': 0.5,
            'stage1_volume_weight': 0.5,
            'stage1_pass_ratio': 0.4
        }
    
    screener = NeglectedSectorScreener(params=params)

    # 2. 기준일자 설정 (현재 기준 가장 최근 영업일로 수정 가능)
    base_date = date(2026, 7, 27) 
    date_1m_ago = base_date - relativedelta(months=1)
    date_6m_ago = base_date - relativedelta(months=6)
    date_1y_ago = base_date - relativedelta(years=1)

    # 3. KOSPI 유니버스 로드
    print("\n📊 1. 유니버스 데이터 로드 중...")
    try:
        universe = loader.get_kospi_universe(base_date)
        print(f" -> KOSPI 유니버스 {len(universe)}종목 로드 완료")
    except Exception as e:
        print(f"❌ 유니버스 로드 실패: {e}")
        return

    # 4. 종목별 OHLCV 취합 및 섹터별 집계
    print("\n📊 2. 개별 종목 OHLCV 캐시 로드 및 섹터 집계 중... (시간이 조금 걸릴 수 있습니다)")
    
    ticker_metrics = []
    market_vol_1m = 0
    market_vol_1y = 0

    for idx, row in universe.iterrows():
        ticker = row['ticker']
        sector = row['sector']
        mcap = row['market_cap']
        
        # 1년치 OHLCV 캐시 로드
        df_ohlcv = loader.get_historical_ohlcv(ticker, date_1y_ago, base_date)
        
        if df_ohlcv is None or df_ohlcv.empty:
            continue
            
        # 거래대금 추정 (종가 * 거래량)
        df_ohlcv['Amount'] = df_ohlcv['Close'] * df_ohlcv['Volume']
        
        mask_1m = df_ohlcv.index >= pd.to_datetime(date_1m_ago)
        mask_6m = df_ohlcv.index >= pd.to_datetime(date_6m_ago)
        
        try:
            # 개별 종목 기간별 수익률
            price_now = df_ohlcv['Close'].iloc[-1]
            price_1m = df_ohlcv.loc[mask_1m, 'Close'].iloc[0]
            price_6m = df_ohlcv.loc[mask_6m, 'Close'].iloc[0]
            
            ret_1m = (price_now / price_1m) - 1
            ret_6m = (price_now / price_6m) - 1
        except IndexError:
            # 상장된 지 1년 미만인 종목 등은 패스
            continue 
            
        # 기간별 거래대금 합산
        vol_1m = df_ohlcv.loc[mask_1m, 'Amount'].sum()
        vol_1y = df_ohlcv['Amount'].sum()
        
        market_vol_1m += vol_1m
        market_vol_1y += vol_1y
        
        ticker_metrics.append({
            'sector': sector,
            'mcap': mcap,
            'ret_1m': ret_1m,
            'ret_6m': ret_6m,
            'vol_1m': vol_1m,
            'vol_1y': vol_1y
        })

    metrics_df = pd.DataFrame(ticker_metrics)

    # 5. 시가총액 가중평균을 이용한 섹터 지표 산출
    print("\n📊 3. 시가총액 가중치 기반 섹터 소외도 점수 계산 중...")
    metrics_df['mcap_weight_1m'] = metrics_df['ret_1m'] * metrics_df['mcap']
    metrics_df['mcap_weight_6m'] = metrics_df['ret_6m'] * metrics_df['mcap']

    sector_grouped = metrics_df.groupby('sector').agg(
        total_mcap=('mcap', 'sum'),
        sum_weight_1m=('mcap_weight_1m', 'sum'),
        sum_weight_6m=('mcap_weight_6m', 'sum'),
        sector_vol_1m=('vol_1m', 'sum'),
        sector_vol_1y=('vol_1y', 'sum'),
        ticker_count=('sector', 'count')
    ).reset_index()

    # 종목수가 너무 적은(예: 3개 미만) 소수 섹터는 노이즈 방지를 위해 제외
    sector_grouped = sector_grouped[sector_grouped['ticker_count'] >= 3].copy()

    # 섹터별 지표 최종 계산
    sector_grouped['return_1m'] = sector_grouped['sum_weight_1m'] / sector_grouped['total_mcap']
    sector_grouped['return_6m'] = sector_grouped['sum_weight_6m'] / sector_grouped['total_mcap']
    sector_grouped['vol_prop_1m'] = sector_grouped['sector_vol_1m'] / market_vol_1m
    sector_grouped['vol_prop_1y'] = sector_grouped['sector_vol_1y'] / market_vol_1y

    final_sector_df = sector_grouped[['sector', 'return_1m', 'return_6m', 'vol_prop_1m', 'vol_prop_1y']]
    
    print("\n[가공된 실제 섹터 데이터 (상위 5개)]")
    display(final_sector_df.head())

    # 6. Stage 1 스크리너 실행
    print("\n🚀 4. Stage 1 필터링 실행 결과")
    passed_sectors = screener.run(final_sector_df)
    display(passed_sectors)

    # 밸류트랩 경고 확인
    value_traps = passed_sectors[passed_sectors['is_value_trap_warning'] == True]
    if not value_traps.empty:
        print(f"\n⚠️ 경고: 다음 섹터는 낙폭이 가팔라지고 있습니다 (Value Trap 위험)\n -> {value_traps['sector'].tolist()}")
    else:
        print("\n✅ 통과된 섹터 중 밸류트랩 경고가 발생한 섹터는 없습니다.")

if __name__ == "__main__":
    run_real_data_test()

🚀 실데이터 기반 Stage 1: 소외 섹터 발굴 테스트

✅ params.yaml 로드 완료

📊 1. 유니버스 데이터 로드 중...
 -> KOSPI 유니버스 833종목 로드 완료

📊 2. 개별 종목 OHLCV 캐시 로드 및 섹터 집계 중... (시간이 조금 걸릴 수 있습니다)

📊 3. 시가총액 가중치 기반 섹터 소외도 점수 계산 중...

[가공된 실제 섹터 데이터 (상위 5개)]


,sector,return_1m,return_6m,vol_prop_1m,vol_prop_1y
0,1차 비철금속 제조업,-0.086696,-0.421099,0.001009,0.005767
1,1차 철강 제조업,-0.039841,-0.132392,0.005262,0.012175
2,가구 제조업,0.021834,-0.302736,0.000082,0.000122
5,"가죽, 가방 및 유사제품 제조업",0.056641,-0.023234,0.000048,0.000098
7,건물 건설업,-0.049687,1.488079,0.023025,0.015387



🚀 4. Stage 1 필터링 실행 결과


,sector,return_z_score,volume_z_score,composite_score,is_value_trap_warning
0,1차 비철금속 제조업,1.051731,0.936994,0.994363,True
114,텔레비전 방송업,0.740268,0.887585,0.813927,False
86,의료용 기기 제조업,0.684017,0.766294,0.725156,True
14,"골판지, 종이 상자 및 종이용기 제조업",0.423264,0.980294,0.701779,False
41,도로 화물 운송업,0.598366,0.790300,0.694333,False
26,기타 금속 가공제품 제조업,0.355765,1.025662,0.690713,False
19,그외 기타 운송장비 제조업,0.934503,0.425236,0.679870,True
53,"비료, 농약 및 살균, 살충제 제조업",0.301031,0.970113,0.635572,False
89,일반 목적용 기계 제조업,0.589506,0.677394,0.633450,True
118,"펄프, 종이 및 판지 제조업",0.624961,0.614323,0.619642,False



⚠️ 경고: 다음 섹터는 낙폭이 가팔라지고 있습니다 (Value Trap 위험)
 -> ['1차 비철금속 제조업', '의료용 기기 제조업', '그외 기타 운송장비 제조업', '일반 목적용 기계 제조업', '의약품 제조업', '내화, 비내화 요업제품 제조업', '플라스틱제품 제조업', '유원지 및 기타 오락관련 서비스업', '선박 및 보트 건조업', '시멘트, 석회, 플라스터 및 그 제품 제조업', '1차 철강 제조업', '자동차용 엔진 및 자동차 제조업', '기초 화학물질 제조업', '소프트웨어 개발 및 공급업']


In [3]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader

print("==================================================")
print("🚀 [검증] 공시 시차(Disclosure Lag) 및 미래참조 방지 테스트")
print("==================================================\n")

loader = QuantDataLoader(use_cache=True)
ticker = '005930' # 삼성전자
year = 2025
quarter = 1

# ---------------------------------------------------------
# 테스트 1: 공시 이전 시점 (Look-ahead bias 발생 가능 시점)
# 1분기 보고서 제출 마감일(보통 5월 15일) 이전인 '4월 30일' 기준
# ---------------------------------------------------------
base_date_before_release = date(2025, 4, 30)
print(f"▶️ 테스트 1: 기준일 = {base_date_before_release} (1Q 실적발표 전)")
data_before = loader.get_isolated_quarterly_financials(ticker, year, quarter, base_date=base_date_before_release)

if pd.isna(data_before.get('revenue')):
    print("✅ 성공: 아직 공시되지 않은 미래의 데이터를 정확히 차단하여 NaN을 반환했습니다.")
else:
    print(f"❌ 실패: 공시 전인데 미래 데이터를 가져왔습니다! 매출액: {data_before.get('revenue')}")

print("-" * 50)

# ---------------------------------------------------------
# 테스트 2: 공시 이후 시점 (정상적인 데이터 수집)
# 1분기 보고서 제출 마감일 이후인 '6월 1일' 기준
# ---------------------------------------------------------
base_date_after_release = date(2025, 6, 1)
print(f"\n▶️ 테스트 2: 기준일 = {base_date_after_release} (1Q 실적발표 후)")
data_after = loader.get_isolated_quarterly_financials(ticker, year, quarter, base_date=base_date_after_release)

if pd.notna(data_after.get('revenue')):
    print(f"✅ 성공: 공시가 완료된 데이터를 정상적으로 불러왔습니다. 매출액: {data_after.get('revenue'):,.0f}")
else:
    print("❌ 실패: 공시 이후임에도 데이터를 가져오지 못했습니다.")
print("\n==================================================")

[미래참조 방지] 005930의 2025년 11013 보고서는 2025-04-30 시점에 미공시 상태입니다. (실제 공시일: 2025-05-15)


🚀 [검증] 공시 시차(Disclosure Lag) 및 미래참조 방지 테스트

▶️ 테스트 1: 기준일 = 2025-04-30 (1Q 실적발표 전)
✅ 성공: 아직 공시되지 않은 미래의 데이터를 정확히 차단하여 NaN을 반환했습니다.
--------------------------------------------------

▶️ 테스트 2: 기준일 = 2025-06-01 (1Q 실적발표 후)
✅ 성공: 공시가 완료된 데이터를 정상적으로 불러왔습니다. 매출액: 79,140,503,000,000

